In [3]:
# DC Housing Market Model Comparison
# This script implements and compares multiple modeling approaches
# for predicting DC housing price changes

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Linear models
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Tree-based models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Time series models
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Set visualization styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

# -------------------------------------------------------------
# 1. Data Loading and Preparation
# -------------------------------------------------------------

def load_and_prepare_data():
    """Load and prepare data for modeling"""
    # Load the clean dataset (for linear models)
    df_clean = pd.read_csv('dc_housing_modeling_features_clean.csv', parse_dates=['Date'], index_col='Date')
    
    # Load the imputed dataset (for tree-based models)
    df_partial = pd.read_csv('dc_housing_modeling_features_partial.csv', parse_dates=['Date'], index_col='Date')
    
    print(f"Clean dataset shape: {df_clean.shape}")
    print(f"Imputed dataset shape: {df_partial.shape}")
    print(f"Clean dataset time range: {df_clean.index.min()} to {df_clean.index.max()}")
    print(f"Imputed dataset time range: {df_partial.index.min()} to {df_partial.index.max()}")
    
    # Define X and y for clean dataset
    X_clean = df_clean.drop('ZHVI_pct_change', axis=1)
    y_clean = df_clean['ZHVI_pct_change']
    
    # Define X and y for imputed dataset
    X_partial = df_partial.drop('ZHVI_pct_change', axis=1)
    y_partial = df_partial['ZHVI_pct_change']
    
    return X_clean, y_clean, X_partial, y_partial

# -------------------------------------------------------------
# 2. Model Evaluation Functions
# -------------------------------------------------------------

def evaluate_model_with_tscv(model, X, y, cv, model_name="Model"):
    """Evaluate model with time series cross-validation"""
    cv_scores_rmse = []
    cv_scores_mae = []
    cv_scores_r2 = []
    cv_directional_accuracy = []
    
    # For plotting actual vs predicted
    all_y_test = []
    all_y_pred = []
    all_test_indices = []
    
    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Store test results for plotting
        all_y_test.extend(y_test)
        all_y_pred.extend(y_pred)
        all_test_indices.extend(test_idx)
        
        # Calculate metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Calculate directional accuracy (new)
        actual_direction = np.sign(y_test)
        pred_direction = np.sign(y_pred)
        direction_match = np.mean(actual_direction == pred_direction)
        cv_directional_accuracy.append(direction_match)
        
        cv_scores_rmse.append(rmse)
        cv_scores_mae.append(mae)
        cv_scores_r2.append(r2)
    
    # Print average metrics
    print(f"{model_name} Cross-validation Results:")
    print(f"RMSE: {np.mean(cv_scores_rmse):.2f} (±{np.std(cv_scores_rmse):.2f})")
    print(f"MAE: {np.mean(cv_scores_mae):.2f} (±{np.std(cv_scores_mae):.2f})")
    print(f"R²: {np.mean(cv_scores_r2):.3f} (±{np.std(cv_scores_r2):.3f})")
    print(f"Directional Accuracy: {np.mean(cv_directional_accuracy):.2%} (±{np.std(cv_directional_accuracy):.2%})")
    
    # Create a DataFrame with test results for plotting
    results_df = pd.DataFrame({
        'Index': X.iloc[all_test_indices].index,
        'Actual': all_y_test,
        'Predicted': all_y_pred
    }).sort_values('Index')
    
    metrics = {
        'rmse': np.mean(cv_scores_rmse),
        'mae': np.mean(cv_scores_mae),
        'r2': np.mean(cv_scores_r2),
        'dir_acc': np.mean(cv_directional_accuracy)
    }
    
    return results_df, metrics

def plot_actual_vs_predicted(results_df, model_name, metrics):
    """Plot actual vs. predicted values"""
    plt.figure(figsize=(14, 7))
    plt.plot(results_df['Index'], results_df['Actual'], marker='o', markersize=6, label='Actual', color='blue', alpha=0.7)
    plt.plot(results_df['Index'], results_df['Predicted'], marker='x', markersize=6, label='Predicted', color='red', alpha=0.7)
    
    # Add regression lines for both series
    actual_trend = np.polyfit(range(len(results_df)), results_df['Actual'], 1)
    pred_trend = np.polyfit(range(len(results_df)), results_df['Predicted'], 1)
    
    plt.plot(results_df['Index'], np.polyval(actual_trend, range(len(results_df))), '--', color='blue', alpha=0.5, label='Actual trend')
    plt.plot(results_df['Index'], np.polyval(pred_trend, range(len(results_df))), '--', color='red', alpha=0.5, label='Predicted trend')
    
    # Add shaded areas for recession periods
    recession_periods = [
        ('2001-03-01', '2001-11-01'),  # Dot-com bubble
        ('2007-12-01', '2009-06-01'),  # Great Recession
        ('2020-02-01', '2020-06-01')   # COVID-19 Recession
    ]
    
    for start, end in recession_periods:
        plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.2, color='gray')
    
    plt.title(f'{model_name}: Actual vs Predicted Housing Price Change (%)', fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Price Change (%)', fontsize=12)
    
    # Add metrics as text
    metrics_text = (f"RMSE: {metrics['rmse']:.2f}\n"
                   f"R²: {metrics['r2']:.3f}\n"
                   f"Dir. Acc: {metrics['dir_acc']:.1%}")
    
    plt.annotate(metrics_text, xy=(0.02, 0.95), xycoords='axes fraction',
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                 fontsize=9, ha='left', va='top')
    
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return plt

def get_feature_importance(model, X, model_type='linear'):
    """Extract feature importance from different model types"""
    if model_type == 'linear':
        # For linear models (Ridge, Lasso, ElasticNet)
        feature_importance = pd.DataFrame({
            'Feature': X.columns,
            'Coefficient': model.coef_,
            'Abs_Coefficient': np.abs(model.coef_)
        })
        feature_importance = feature_importance.sort_values('Abs_Coefficient', ascending=False)
    
    elif model_type == 'tree':
        # For tree-based models (RF, GBM, XGBoost)
        if hasattr(model, 'feature_importances_'):
            # Random Forest and Gradient Boosting
            feature_importance = pd.DataFrame({
                'Feature': X.columns,
                'Importance': model.feature_importances_,
            })
        else:
            # XGBoost
            feature_importance = pd.DataFrame({
                'Feature': X.columns,
                'Importance': model.get_booster().get_score(importance_type='gain')
            })
        feature_importance = feature_importance.sort_values('Importance', ascending=False)
    
    return feature_importance

def plot_feature_importance(feature_importance, model_name, importance_type='linear'):
    """Plot feature importance for different model types"""
    plt.figure(figsize=(14, 10))
    
    if importance_type == 'linear':
        # Limit to top 15 features for readability
        top_features = feature_importance.head(15)
        bars = plt.barh(top_features['Feature'], top_features['Coefficient'])
        
        # Color the bars based on the sign of the coefficient
        for i, bar in enumerate(bars):
            if top_features.iloc[i]['Coefficient'] < 0:
                bar.set_color('red')
            else:
                bar.set_color('green')
        
        plt.title(f'{model_name}: Feature Importance (Coefficients)', fontsize=14)
        plt.xlabel('Coefficient Value', fontsize=12)
        plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    
    else:
        # For tree-based models
        # Limit to top 15 features for readability
        top_features = feature_importance.head(15)
        plt.barh(top_features['Feature'], top_features['Importance'], color='skyblue')
        plt.title(f'{model_name}: Feature Importance', fontsize=14)
        plt.xlabel('Importance', fontsize=12)
    
    plt.ylabel('Feature', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return plt

# -------------------------------------------------------------
# 3. Linear Models Implementation
# -------------------------------------------------------------

def implement_linear_models(X, y):
    """Implement and evaluate linear models with time series cross-validation"""
    results = {}
    tscv = TimeSeriesSplit(n_splits=2)  # Use 2-fold cross-validation as found optimal
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    
    # 1. Ridge Regression
    print("\n===== RIDGE REGRESSION =====")
    ridge_params = {'alpha': [50, 60, 70, 73, 75, 80, 90, 100]}
    grid_ridge = GridSearchCV(Ridge(), ridge_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_ridge.fit(X_scaled, y)
    best_alpha = grid_ridge.best_params_['alpha']
    print(f"Best alpha: {best_alpha}")
    
    ridge_model = Ridge(alpha=best_alpha)
    ridge_results, ridge_metrics = evaluate_model_with_tscv(ridge_model, X_scaled, y, tscv, "Ridge Regression")
    ridge_importance = get_feature_importance(Ridge(alpha=best_alpha).fit(X_scaled, y), X, 'linear')
    
    ridge_plot = plot_actual_vs_predicted(ridge_results, f"Ridge (α={best_alpha})", ridge_metrics)
    ridge_importance_plot = plot_feature_importance(ridge_importance, f"Ridge (α={best_alpha})", 'linear')
    
    results['Ridge'] = {
        'model': ridge_model,
        'params': {'alpha': best_alpha},
        'results': ridge_results,
        'metrics': ridge_metrics,
        'importance': ridge_importance,
        'plots': {'prediction': ridge_plot, 'importance': ridge_importance_plot}
    }
    
    # 2. Lasso Regression
    print("\n===== LASSO REGRESSION =====")
    lasso_params = {'alpha': [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]}
    grid_lasso = GridSearchCV(Lasso(max_iter=10000), lasso_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_lasso.fit(X_scaled, y)
    best_lasso_alpha = grid_lasso.best_params_['alpha']
    print(f"Best alpha: {best_lasso_alpha}")
    
    lasso_model = Lasso(alpha=best_lasso_alpha, max_iter=10000)
    lasso_results, lasso_metrics = evaluate_model_with_tscv(lasso_model, X_scaled, y, tscv, "Lasso Regression")
    lasso_importance = get_feature_importance(Lasso(alpha=best_lasso_alpha, max_iter=10000).fit(X_scaled, y), X, 'linear')
    
    lasso_plot = plot_actual_vs_predicted(lasso_results, f"Lasso (α={best_lasso_alpha})", lasso_metrics)
    lasso_importance_plot = plot_feature_importance(lasso_importance, f"Lasso (α={best_lasso_alpha})", 'linear')
    
    results['Lasso'] = {
        'model': lasso_model,
        'params': {'alpha': best_lasso_alpha},
        'results': lasso_results,
        'metrics': lasso_metrics,
        'importance': lasso_importance,
        'plots': {'prediction': lasso_plot, 'importance': lasso_importance_plot}
    }
    
    # 3. Elastic Net
    print("\n===== ELASTIC NET =====")
    # Grid search for both alpha and l1_ratio
    enet_params = {
        'alpha': [0.05, 0.1, 0.3, 0.5, 0.7, 1.0, 2.0],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    }
    grid_enet = GridSearchCV(ElasticNet(max_iter=10000), enet_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_enet.fit(X_scaled, y)
    best_enet_params = grid_enet.best_params_
    print(f"Best parameters: {best_enet_params}")
    
    enet_model = ElasticNet(alpha=best_enet_params['alpha'], l1_ratio=best_enet_params['l1_ratio'], max_iter=10000)
    enet_results, enet_metrics = evaluate_model_with_tscv(enet_model, X_scaled, y, tscv, "Elastic Net")
    enet_importance = get_feature_importance(
        ElasticNet(alpha=best_enet_params['alpha'], l1_ratio=best_enet_params['l1_ratio'], max_iter=10000).fit(X_scaled, y), 
        X, 'linear'
    )
    
    enet_plot = plot_actual_vs_predicted(enet_results, f"Elastic Net (α={best_enet_params['alpha']}, L1={best_enet_params['l1_ratio']})", enet_metrics)
    enet_importance_plot = plot_feature_importance(enet_importance, f"Elastic Net (α={best_enet_params['alpha']}, L1={best_enet_params['l1_ratio']})", 'linear')
    
    results['ElasticNet'] = {
        'model': enet_model,
        'params': best_enet_params,
        'results': enet_results,
        'metrics': enet_metrics,
        'importance': enet_importance,
        'plots': {'prediction': enet_plot, 'importance': enet_importance_plot}
    }
    
    return results

# -------------------------------------------------------------
# 4. Tree-based Models Implementation
# -------------------------------------------------------------

def implement_tree_models(X, y):
    """Implement and evaluate tree-based models with time series cross-validation"""
    results = {}
    tscv = TimeSeriesSplit(n_splits=2)  # Use 2-fold cross-validation as found optimal
    
    # 1. Random Forest
    print("\n===== RANDOM FOREST =====")
    rf_params = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 3, 5, 7, 10],
        'min_samples_split': [2, 5, 10]
    }
    grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_rf.fit(X, y)
    best_rf_params = grid_rf.best_params_
    print(f"Best parameters: {best_rf_params}")
    
    rf_model = RandomForestRegressor(
        n_estimators=best_rf_params['n_estimators'],
        max_depth=best_rf_params['max_depth'],
        min_samples_split=best_rf_params['min_samples_split'],
        random_state=42
    )
    rf_results, rf_metrics = evaluate_model_with_tscv(rf_model, X, y, tscv, "Random Forest")
    rf_importance = get_feature_importance(
        RandomForestRegressor(
            n_estimators=best_rf_params['n_estimators'],
            max_depth=best_rf_params['max_depth'],
            min_samples_split=best_rf_params['min_samples_split'],
            random_state=42
        ).fit(X, y), 
        X, 'tree'
    )
    
    rf_plot = plot_actual_vs_predicted(rf_results, "Random Forest", rf_metrics)
    rf_importance_plot = plot_feature_importance(rf_importance, "Random Forest", 'tree')
    
    results['RandomForest'] = {
        'model': rf_model,
        'params': best_rf_params,
        'results': rf_results,
        'metrics': rf_metrics,
        'importance': rf_importance,
        'plots': {'prediction': rf_plot, 'importance': rf_importance_plot}
    }
    
    # 2. Gradient Boosting
    print("\n===== GRADIENT BOOSTING =====")
    gb_params = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [2, 3, 4, 5],
        'subsample': [0.8, 1.0]
    }
    grid_gb = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_gb.fit(X, y)
    best_gb_params = grid_gb.best_params_
    print(f"Best parameters: {best_gb_params}")
    
    gb_model = GradientBoostingRegressor(
        n_estimators=best_gb_params['n_estimators'],
        learning_rate=best_gb_params['learning_rate'],
        max_depth=best_gb_params['max_depth'],
        subsample=best_gb_params['subsample'],
        random_state=42
    )
    gb_results, gb_metrics = evaluate_model_with_tscv(gb_model, X, y, tscv, "Gradient Boosting")
    gb_importance = get_feature_importance(
        GradientBoostingRegressor(
            n_estimators=best_gb_params['n_estimators'],
            learning_rate=best_gb_params['learning_rate'],
            max_depth=best_gb_params['max_depth'],
            subsample=best_gb_params['subsample'],
            random_state=42
        ).fit(X, y), 
        X, 'tree'
    )
    
    gb_plot = plot_actual_vs_predicted(gb_results, "Gradient Boosting", gb_metrics)
    gb_importance_plot = plot_feature_importance(gb_importance, "Gradient Boosting", 'tree')
    
    results['GradientBoosting'] = {
        'model': gb_model,
        'params': best_gb_params,
        'results': gb_results,
        'metrics': gb_metrics,
        'importance': gb_importance,
        'plots': {'prediction': gb_plot, 'importance': gb_importance_plot}
    }
    
    # 3. XGBoost
    print("\n===== XGBOOST =====")
    xgb_params = {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 4, 5, 6],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    }
    grid_xgb = GridSearchCV(xgb.XGBRegressor(random_state=42), xgb_params, cv=tscv, scoring='neg_mean_squared_error')
    grid_xgb.fit(X, y)
    best_xgb_params = grid_xgb.best_params_
    print(f"Best parameters: {best_xgb_params}")
    
    xgb_model = xgb.XGBRegressor(
        n_estimators=best_xgb_params['n_estimators'],
        learning_rate=best_xgb_params['learning_rate'],
        max_depth=best_xgb_params['max_depth'],
        subsample=best_xgb_params['subsample'],
        colsample_bytree=best_xgb_params['colsample_bytree'],
        random_state=42
    )
    xgb_results, xgb_metrics = evaluate_model_with_tscv(xgb_model, X, y, tscv, "XGBoost")
    
    # For XGBoost, we need to handle feature importance differently
    xgb_fitted = xgb.XGBRegressor(
        n_estimators=best_xgb_params['n_estimators'],
        learning_rate=best_xgb_params['learning_rate'],
        max_depth=best_xgb_params['max_depth'],
        subsample=best_xgb_params['subsample'],
        colsample_bytree=best_xgb_params['colsample_bytree'],
        random_state=42
    ).fit(X, y)
    
    # Get feature importance
    xgb_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': xgb_fitted.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    xgb_plot = plot_actual_vs_predicted(xgb_results, "XGBoost", xgb_metrics)
    xgb_importance_plot = plot_feature_importance(xgb_importance, "XGBoost", 'tree')
    
    results['XGBoost'] = {
        'model': xgb_model,
        'params': best_xgb_params,
        'results': xgb_results,
        'metrics': xgb_metrics,
        'importance': xgb_importance,
        'plots': {'prediction': xgb_plot, 'importance': xgb_importance_plot}
    }
    
    return results

# -------------------------------------------------------------
# 5. Time Series Models Implementation
# -------------------------------------------------------------

def implement_time_series_models(X, y):
    """Implement and evaluate time series specific models"""
    results = {}
    tscv = TimeSeriesSplit(n_splits=2)  # Use 2-fold cross-validation as found optimal
    
    # Function to make predictions with ARIMAX models
    def predict_arimax(model, exog_train, exog_test, endog_train, steps):
        """Make multi-step predictions with ARIMAX models"""
        model_fit = model.fit(exog=exog_train, endog=endog_train)
        return model_fit.forecast(steps=steps, exog=exog_test)
    
    # SARIMAX model - this uses exogenous variables (X)
    print("\n===== SARIMAX MODEL =====")
    
    # For SARIMAX, we need to standardize the exogenous variables
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    
    # We'll use a simplified grid search for SARIMAX due to computational complexity
    best_aic = float('inf')
    best_order = None
    best_seasonal_order = None
    
    # Define a smaller grid for SARIMAX
    p_values = [0, 1, 2]
    d_values = [1]  # We know the series needs differencing
    q_values = [0, 1]
    P_values = [0]
    D_values = [0]
    Q_values = [0]
    s_values = [0]  # No seasonality for quarterly data
    
    # Store cross-validation results
    all_y_test = []
    all_y_pred = []
    all_test_indices = []
    
    for train_idx, test_idx in tscv.split(X_scaled):
        X_train, X_test = X_scaled.iloc[train_idx], X_scaled.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Select a subset of features to avoid high dimensionality issues
        # Use the most important features from previous models
        important_features = ['hot_market', 'recession_period', 'Interest_Rate_diff1', 
                            'CPI', 'GDP_Growth', 'Unemployment_Rate', 'real_mortgage_rate']
        X_train_selected = X_train[important_features]
        X_test_selected = X_test[important_features]
        
        best_aic_fold = float('inf')
        best_order_fold = None
        best_seasonal_order_fold = None
        
        # Grid search for SARIMAX parameters (simplified)
        for p in p_values:
            for d in d_values:
                for q in q_values:
                    for P in P_values:
                        for D in D_values:
                            for Q in Q_values:
                                for s in s_values:
                                    if s == 0 and (P > 0 or D > 0 or Q > 0):
                                        continue  # Skip seasonal parameters if s=0
                                        
                                    order = (p, d, q)
                                    seasonal_order = (P, D, Q, s)
                                    
                                    try:
                                        model = SARIMAX(
                                            y_train, 
                                            exog=X_train_selected, 
                                            order=order,
                                            seasonal_order=seasonal_order
                                        )
                                        model_fit = model.fit(disp=False)
                                        aic = model_fit.aic
                                        
                                        if aic < best_aic_fold:
                                            best_aic_fold = aic
                                            best_order_fold = order
                                            best_seasonal_order_fold = seasonal_order
                                    except:
                                        continue
        
        if best_order_fold is not None:
            print(f"Fold best order: {best_order_fold}, best seasonal order: {best_seasonal_order_fold}")
            
            # Use the best model for this fold to make predictions
            model = SARIMAX(
                y_train, 
                exog=X_train_selected, 
                order=best_order_fold,
                seasonal_order=best_seasonal_order_fold
            )
            model_fit = model.fit(disp=False)
            
            # Forecast the test period
            forecasts = model_fit.forecast(steps=len(X_test_selected), exog=X_test_selected)
            
            # Store results
            all_y_test.extend(y_test)
            all_y_pred.extend(forecasts)
            all_test_indices.extend(test_idx)
            
            # Update the best model parameters if this fold's model is better
            if best_aic_fold < best_aic:
                best_aic = best_aic_fold
                best_order = best_order_fold
                best_seasonal_order = best_seasonal_order_fold
    
    # Create a DataFrame with test results for plotting
    sarimax_results = pd.DataFrame({
        'Index': X.iloc[all_test_indices].index,
        'Actual': all_y_test,
        'Predicted': all_y_pred
    }).sort_values('Index')
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(sarimax_results['Actual'], sarimax_results['Predicted']))
    mae = mean_absolute_error(sarimax_results['Actual'], sarimax_results['Predicted'])
    r2 = r2_score(sarimax_results['Actual'], sarimax_results['Predicted'])
    
    # Calculate directional accuracy
    actual_direction = np.sign(sarimax_results['Actual'])
    pred_direction = np.sign(sarimax_results['Predicted'])
    dir_acc = np.mean(actual_direction == pred_direction)
    
    sarimax_metrics = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'dir_acc': dir_acc
    }
    
    print(f"SARIMAX Final Results:")
    print(f"Best order: {best_order}, best seasonal order: {best_seasonal_order}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.3f}")
    print(f"Directional Accuracy: {dir_acc:.2%}")
    
    sarimax_plot = plot_actual_vs_predicted(sarimax_results, f"SARIMAX {best_order}", sarimax_metrics)
    
    results['SARIMAX'] = {
        'params': {'order': best_order, 'seasonal_order': best_seasonal_order},
        'results': sarimax_results,
        'metrics': sarimax_metrics,
        'plots': {'prediction': sarimax_plot}
    }
    
    return results

# -------------------------------------------------------------
# 7. Model Comparison and Evaluation
# -------------------------------------------------------------

def compare_models(all_results):
    """Compare all models and visualize their performance"""
    # Extract metrics from all models
    comparison = {}
    
    for model_type, models in all_results.items():
        for model_name, model_data in models.items():
            comparison[model_name] = model_data['metrics']
    
    # Convert to DataFrame for easy visualization
    comparison_df = pd.DataFrame.from_dict(comparison, orient='index')
    comparison_df.sort_values('rmse', inplace=True)
    
    # Plot comparison of metrics
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # RMSE (lower is better)
    axs[0, 0].bar(comparison_df.index, comparison_df['rmse'], color='skyblue')
    axs[0, 0].set_title('RMSE (lower is better)', fontsize=14)
    axs[0, 0].set_ylim(bottom=0)
    axs[0, 0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['rmse']):
        axs[0, 0].text(i, v + 0.1, f"{v:.2f}", ha='center')
    
    # MAE (lower is better)
    axs[0, 1].bar(comparison_df.index, comparison_df['mae'], color='lightgreen')
    axs[0, 1].set_title('MAE (lower is better)', fontsize=14)
    axs[0, 1].set_ylim(bottom=0)
    axs[0, 1].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['mae']):
        axs[0, 1].text(i, v + 0.1, f"{v:.2f}", ha='center')
    
    # R² (higher is better)
    axs[1, 0].bar(comparison_df.index, comparison_df['r2'], color='salmon')
    axs[1, 0].set_title('R² (higher is better)', fontsize=14)
    axs[1, 0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['r2']):
        axs[1, 0].text(i, v + 0.05, f"{v:.3f}", ha='center')
    
    # Directional Accuracy (higher is better)
    axs[1, 1].bar(comparison_df.index, comparison_df['dir_acc'], color='plum')
    axs[1, 1].set_title('Directional Accuracy (higher is better)', fontsize=14)
    axs[1, 1].set_ylim(0, 1)
    axs[1, 1].tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df['dir_acc']):
        axs[1, 1].text(i, v + 0.02, f"{v:.1%}", ha='center')
    
    plt.tight_layout()
    plt.show()
    
    # Create a summary table
    print("\n===== MODEL PERFORMANCE COMPARISON =====")
    print(comparison_df.sort_values('rmse').to_string())
    
    # Identify best model for each metric
    best_rmse = comparison_df.sort_values('rmse').index[0]
    best_r2 = comparison_df.sort_values('r2', ascending=False).index[0]
    best_dir_acc = comparison_df.sort_values('dir_acc', ascending=False).index[0]
    
    print("\nBest models by metric:")
    print(f"Best RMSE: {best_rmse} ({comparison_df.loc[best_rmse, 'rmse']:.2f})")
    print(f"Best R²: {best_r2} ({comparison_df.loc[best_r2, 'r2']:.3f})")
    print(f"Best Directional Accuracy: {best_dir_acc} ({comparison_df.loc[best_dir_acc, 'dir_acc']:.1%})")
    
    return comparison_df

# -------------------------------------------------------------
# 8. Main Execution Function
# -------------------------------------------------------------

def main():
    """Main execution function"""
    print("==== DC HOUSING MARKET MODEL COMPARISON ====")
    
    # 1. Load and prepare data
    X_clean, y_clean, X_partial, y_partial = load_and_prepare_data()
    
    # 2. Implement linear models (using clean dataset)
    linear_results = implement_linear_models(X_clean, y_clean)
    
    # 3. Implement tree-based models (using imputed dataset)
    tree_results = implement_tree_models(X_partial, y_partial)
    
    # 4. Implement time series models (using clean dataset)
    ts_results = implement_time_series_models(X_clean, y_clean)